In [ ]:
"""
МХБ-ийн хувьцааны өдрийн арилжааны ТҮҮХИЙГ CSV болгож татах.
(mse_top20.py-ийн задлагчийг ашигласан, цувааны шинжилгээнд зориулав.)

Ажиллуулах:
    python mse_history.py APU 2010-01-01 2026-10-14
    python mse_history.py APU,TTL,KHAN 2015-01-01 2026-10-14
    python mse_history.py ALL 2010-01-01 2026-10-14      # TOP-20 бүгд
Spyder/Jupyter-оос: доорх TICKERS, START_DATE, END_DATE-ийг засаад Run.

Гаралт:
    mse_<TICKER>_<start>_<end>.csv   (date, open, high, low, close, volume, value, deals)
    mse_quality_<start>_<end>.csv    (хувьцаа бүрийн өгөгдлийн чанарын хураангуй)

Шаардлагатай сан:  pip install requests pandas lxml
"""
import io
import sys
import time
import datetime as dt

import pandas as pd
import requests

TRADE_URL = "https://open.mse.mn/securities/{id}/tab/tradeinfo"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

TOP20 = {
    "AARD": "326", "APU": "90", "CUMN": "557", "GLMT": "562", "GOV": "354",
    "INV": "553", "KHAN": "563", "LEND": "545", "MGLA": "572", "MLG": "569",
    "MNDL": "547", "MSE": "510", "QPAY": "570", "SBM": "564", "SUU": "135",
    "TDB": "567", "TTL": "458", "TUM": "549", "UID": "484", "XAC": "568",
}

RENAME = {
    "Огноо": "date", "Нээлтийн ханш": "open", "Хаалтын ханш": "close",
    "Өдрийн хязгаар": "range", "Арилжигдсан ширхэг": "volume",
    "Арилжааны үнийн дүн": "value", "Хэлцлийн тоо": "deals",
}
# --- Spyder/Jupyter-оос ажиллуулах бол эдгээрийг засаад Run дар ---
TICKERS = "ALL"            # "APU", "APU,TTL,KHAN" эсвэл "ALL"
START_DATE = "2010-01-01"
END_DATE = "2026-10-14"

COLS = ["date", "open", "high", "low", "close", "volume", "value", "deals"]


def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace(",", "", regex=False)
         .str.replace("₮", "", regex=False).str.strip(),
        errors="coerce",
    )


def parse_trade_html(html: str) -> pd.DataFrame:
    try:
        tables = pd.read_html(io.StringIO(html), displayed_only=False)
    except ValueError:
        return pd.DataFrame(columns=COLS)
    df = next((t for t in tables
               if {"Огноо", "Хаалтын ханш"} <= {str(c).strip() for c in t.columns}), None)
    if df is None:
        return pd.DataFrame(columns=COLS)

    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns=RENAME)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"]).drop_duplicates(subset=["date"])
    for c in ("open", "close", "volume", "value", "deals"):
        if c in df.columns:
            df[c] = to_num(df[c])
    if "range" in df.columns:
        parts = df["range"].astype(str).str.split(" - ", n=1, expand=True)
        df["low"] = to_num(parts[0])
        df["high"] = to_num(parts[1]) if parts.shape[1] > 1 else df["low"]
    for c in COLS:
        if c not in df.columns:
            df[c] = pd.NA
    return df[COLS].sort_values("date").reset_index(drop=True)


def fetch(session, comp_id):
    r = session.get(TRADE_URL.format(id=comp_id), headers=HEADERS, timeout=(5, 60))
    r.raise_for_status()
    r.encoding = "utf-8"
    return parse_trade_html(r.text)


def quality(ticker, df, start, end):
    """Цувааны шинжилгээнд хэрэгтэй өгөгдлийн чанарын үзүүлэлтүүд."""
    if df.empty:
        return {"ticker": ticker, "rows": 0}
    bdays = pd.bdate_range(df["date"].min(), df["date"].max())   # ажлын өдөр (баярыг тооцоогүй)
    ret = df["close"].pct_change()
    gaps = df["date"].diff().dt.days
    big = ret.abs() > 0.3                                          # хуваалт/алдаа сэжигтэй
    return {
        "ticker": ticker,
        "rows": len(df),
        "first_date": df["date"].min().date(),
        "last_date": df["date"].max().date(),
        "pct_bdays_traded": round(100 * len(df) / len(bdays), 1),
        "pct_zero_return": round(100 * (ret == 0).mean(), 1),
        "max_gap_days": int(gaps.max()) if len(df) > 1 else 0,
        "n_abs_ret_gt_30pct": int(big.sum()),
        "dates_abs_ret_gt_30pct": ";".join(df.loc[big, "date"].dt.strftime("%Y-%m-%d")),
        "missing_close": int(df["close"].isna().sum()),
    }


def main():
    args = [a for a in sys.argv[1:] if not a.startswith("-") and not a.endswith(".json")]
    if len(args) == 3:                       # командын мөрөөс
        tick_arg, s, e = args
    else:                                    # Spyder / Jupyter: файлын дээд талын утгууд
        tick_arg, s, e = TICKERS, START_DATE, END_DATE
    print(f"[+] {tick_arg}: {s} -> {e}")
    start, end = pd.Timestamp(s), pd.Timestamp(e)
    if start > end:
        raise SystemExit("[!] Эхлэх огноо дуусах огнооноос хойно байна.")
    tickers = sorted(TOP20) if tick_arg.upper() == "ALL" else [t.strip().upper() for t in tick_arg.split(",")]

    rows = []
    with requests.Session() as sess:
        for i, t in enumerate(tickers, 1):
            if t not in TOP20:
                print(f"[{i}/{len(tickers)}] {t}: ID мэдэгдэхгүй (TOP20 толь бичигт нэм)")
                continue
            try:
                df = fetch(sess, TOP20[t])
            except requests.RequestException as ex:
                print(f"[{i}/{len(tickers)}] {t}: сүлжээний алдаа — {ex}")
                continue
            full_first = df["date"].min() if not df.empty else None
            df = df[(df["date"] >= start) & (df["date"] <= end)]
            fn = f"mse_{t}_{s}_{e}.csv"
            df.to_csv(fn, index=False, date_format="%Y-%m-%d")
            q = quality(t, df, start, end)
            rows.append(q)
            note = ""
            if full_first is not None and full_first > start:
                note = f"  (сайт {full_first.date()}-ээс эхэлсэн өгөгдөл өгсөн)"
            print(f"[{i}/{len(tickers)}] {t}: {len(df)} мөр -> {fn}{note}")
            time.sleep(1)  # сайтад ачаалал өгөхгүй

    if rows:
        qdf = pd.DataFrame(rows)
        qfn = f"mse_quality_{s}_{e}.csv"
        qdf.to_csv(qfn, index=False)
        pd.set_option("display.width", 200)
        print("\n=== ӨГӨГДЛИЙН ЧАНАР ===")
        print(qdf.drop(columns=["dates_abs_ret_gt_30pct"]).to_string(index=False))
        print(f"\n[+] {qfn}")


if __name__ == "__main__":
    main()

[+] ALL: 2010-01-01 -> 2026-10-14
[1/20] AARD: 2077 мөр -> mse_AARD_2010-01-01_2026-10-14.csv
[2/20] APU: 3407 мөр -> mse_APU_2010-01-01_2026-10-14.csv
[3/20] CUMN: 1216 мөр -> mse_CUMN_2010-01-01_2026-10-14.csv  (сайт 2021-11-15-ээс эхэлсэн өгөгдөл өгсөн)
[4/20] GLMT: 952 мөр -> mse_GLMT_2010-01-01_2026-10-14.csv  (сайт 2022-12-09-ээс эхэлсэн өгөгдөл өгсөн)
[5/20] GOV: 3264 мөр -> mse_GOV_2010-01-01_2026-10-14.csv
[6/20] INV: 1808 мөр -> mse_INV_2010-01-01_2026-10-14.csv  (сайт 2019-06-10-ээс эхэлсэн өгөгдөл өгсөн)
[7/20] KHAN: 855 мөр -> mse_KHAN_2010-01-01_2026-10-14.csv  (сайт 2023-05-01-ээс эхэлсэн өгөгдөл өгсөн)
[8/20] LEND: 2132 мөр -> mse_LEND_2010-01-01_2026-10-14.csv  (сайт 2018-03-14-ээс эхэлсэн өгөгдөл өгсөн)
[9/20] MGLA: 459 мөр -> mse_MGLA_2010-01-01_2026-10-14.csv  (сайт 2024-06-03-ээс эхэлсэн өгөгдөл өгсөн)
[10/20] MLG: 732 мөр -> mse_MLG_2010-01-01_2026-10-14.csv  (сайт 2023-09-27-ээс эхэлсэн өгөгдөл өгсөн)
[11/20] MNDL: 1971 мөр -> mse_MNDL_2010-01-01_2026-10-14.csv  

In [2]:
"""
МХБ-ийн TOP-20 хувьцааны цувааны шинжилгээ (mse_history.py-ийн CSV-үүд дээр).

Ажиллуулах (CSV-үүд байгаа хавтсанд):
    python mse_analysis.py 2010-01-01 2026-10-14
Spyder/Jupyter: доорх START_DATE, END_DATE-ийг засаад Run.

Гаралт (./report/ хавтсанд):
    01_quality.csv       өгөгдлийн шалгалт: үсрэлт, зогсонги ханш, high/low-ийн утга
    02_liquidity.csv     хөрвөх чадвар: арилжаалагдсан өдөр, дундаж дүн, Амихуд, Ролл
    03_returns.csv       өгөөжийн тархалт: хэлбэлзэл, хазайлт, эксцесс, хамгийн их уналт
    04_dependence.csv    автокорреляц, Льюнг–Бокс, дисперсийн харьцаа, |r|-ийн кластер
    05_weekday.csv       гарагийн нөлөө
    06_corr.csv          синхрон корреляцын матриц (зөвхөн хоёулаа арилжаалагдсан өдрүүд)
    07_pca.csv           PCA: зах зээлийн хүчин зүйлийн жин
    08_beta.csv          энгийн ба Димсоны бета (сийрэг арилжааны засвартай)
    *.png                график

Шаардлагатай сан:  pip install pandas numpy scipy matplotlib
"""
import glob
import os
import sys

import numpy as np
import pandas as pd
from scipy import stats

START_DATE = "2010-01-01"
END_DATE = "2026-10-14"
MIN_OBS = 250            # үүнээс цөөн арилжаатай хувьцааг цувааны шинжилгээнээс хасна
JUMP = 0.30              # |өгөөж| > 30% бол хуваалт/алдаа сэжигтэй
OUT = "report"
LIQ_VALUE = 5e6         # индексийн гишүүн: өдрийн медиан арилжаа ≥ 5 сая ₮
LIQ_DAYS = 90           # ба арилжаалагдсан өдөр ≥ 90%


# ---------------------------------------------------------------- ачаалах
def load(start, end):
    data = {}
    for fn in sorted(glob.glob(f"mse_*_{start}_{end}.csv")):
        t = os.path.basename(fn).split("_")[1]
        if t == "quality":
            continue
        df = pd.read_csv(fn, parse_dates=["date"]).sort_values("date")
        df = df.dropna(subset=["close"]).drop_duplicates("date")
        df = df[df["close"] > 0]
        if len(df):
            data[t] = df.set_index("date")
    return data


# ---------------------------------------------------------------- 1. чанар
def quality(t, df):
    c = df["close"]
    r = np.log(c).diff()
    gap = df.index.to_series().diff().dt.days
    hl_ok = df[["high", "low"]].notna().all(axis=1)
    inside = ((df["low"] <= df[["open", "close"]].min(axis=1)) &
              (df["high"] >= df[["open", "close"]].max(axis=1)))[hl_ok]
    width = ((df["high"] - df["low"]) / df["close"])[hl_ok]
    return {
        "ticker": t, "rows": len(df),
        "first": df.index.min().date(), "last": df.index.max().date(),
        "median_gap_days": gap.median(), "max_gap_days": gap.max(),
        "n_jumps_gt30pct": int((r.abs() > np.log(1 + JUMP)).sum()),
        "pct_zero_return": round(100 * (r == 0).mean(), 1),
        "n_weekend_rows": int((df.index.dayofweek >= 5).sum()),
        "pct_zero_volume": round(100 * (df["volume"].fillna(0) == 0).mean(), 1),
        # high/low нь бодит арилжааны хүрээ бол open, close үргэлж дотор нь байна;
        # үнийн хязгаарын (±%) зурвас бол өргөн нь бараг тогтмол байна
        "pct_oc_inside_hl": round(100 * inside.mean(), 1) if len(inside) else np.nan,
        "hl_width_median": round(width.median(), 4) if len(width) else np.nan,
        "hl_width_cv": round(width.std() / width.mean(), 3) if len(width) and width.mean() else np.nan,
    }


# ---------------------------------------------------------------- 2. хөрвөх чадвар
def liquidity(t, df, bdays):
    r = df["close"].pct_change()
    val = df["value"].astype(float)
    amihud = (r.abs() / val.replace(0, np.nan)).dropna()
    dc = np.log(df["close"]).diff()                 # лог үнээр: үнийн түвшин өөрчлөгдөхөд тогтвортой
    cov = dc.cov(dc.shift(1))
    roll = 2 * np.sqrt(-cov) if cov < 0 else np.nan   # Ролл: cov ≥ 0 бол тодорхойлогдохгүй
    return {
        "ticker": t,
        "pct_days_traded": round(100 * len(df) / bdays, 1),
        "median_value_mnt": float(val.median()),
        "median_deals": float(df["deals"].median()),
        "amihud_x1e9": round(1e9 * amihud.median(), 3) if len(amihud) else np.nan,
        "roll_spread_pct": round(100 * roll, 2),
    }


# ---------------------------------------------------------------- 3. өгөөж
def returns_stats(t, r, n_years):
    mdd = (np.exp(r.cumsum()) / np.exp(r.cumsum()).cummax() - 1).min()
    ann = len(r) / n_years                     # жилд арилжаалагдсан өдрийн бодит тоо
    return {
        "ticker": t, "n": len(r), "obs_per_year": round(ann, 1),
        "ann_return_log": round(r.mean() * ann, 4),
        "ann_vol": round(r.std() * np.sqrt(ann), 4),
        "skew": round(stats.skew(r), 2), "excess_kurt": round(stats.kurtosis(r), 2),
        "jarque_bera_p": stats.jarque_bera(r).pvalue,
        "max_drawdown": round(mdd, 3),
        "pct_abs_gt_3sd": round(100 * (np.abs(r - r.mean()) > 3 * r.std()).mean(), 2),
    }


# ---------------------------------------------------------------- 4. хамаарал
def ljung_box(x, lags=10):
    x = x - x.mean(); n = len(x); d = (x ** 2).sum()
    rho = np.array([(x[k:] * x[:-k]).sum() / d for k in range(1, lags + 1)])
    q = n * (n + 2) * np.sum(rho ** 2 / (n - np.arange(1, lags + 1)))
    return rho, q, 1 - stats.chi2.cdf(q, lags)


def variance_ratio(r, q):
    """Ло–МакКинлэйгийн дисперсийн харьцаа, гетероскедастик тэсвэртэй z."""
    r = r.values; n = len(r); mu = r.mean()
    rq = pd.Series(r).rolling(q).sum().dropna().values
    vr = np.sum((rq - q * mu) ** 2) / (n * q) / (np.sum((r - mu) ** 2) / n)
    d = (r - mu) ** 2
    theta = sum((2 * (q - j) / q) ** 2 *
                np.sum(d[j:] * d[:-j]) / np.sum(d) ** 2 * n for j in range(1, q))
    return vr, np.sqrt(n) * (vr - 1) / np.sqrt(theta) if theta > 0 else np.nan


def dependence(t, r):
    rho, q, p = ljung_box(r.values, 10)
    rho_a, qa, pa = ljung_box(np.abs(r.values), 10)
    out = {"ticker": t, "acf1": round(rho[0], 3), "acf2": round(rho[1], 3),
           "ljungbox10_p": p, "acf1_abs": round(rho_a[0], 3), "ljungbox10_abs_p": pa}
    for q_ in (2, 5, 10):
        vr, z = variance_ratio(r, q_)
        out[f"VR{q_}"], out[f"VR{q_}_z"] = round(vr, 3), round(z, 2)
    return out


def weekday(t, r):
    g = r.groupby(r.index.dayofweek)
    res = {"ticker": t}
    for d, s in g:
        name = ["Да", "Мя", "Лх", "Пү", "Ба", "Бя", "Ня"][d]
        res[name] = round(10000 * s.mean(), 1)          # б.н. (0.01%)
        if d >= 5:
            res[f"n_{name}"] = len(s)
    # амралтын өдрийн цөөн ажиглалтыг тестэд оруулахгүй
    res["kruskal_p"] = stats.kruskal(*[s.values for d, s in g if d < 5 and len(s) > 30]).pvalue
    return res


# ---------------------------------------------------------------- 6–8. хөндлөн огтлол
def nearest_psd(c):
    w, v = np.linalg.eigh(c)
    c2 = v @ np.diag(np.clip(w, 1e-8, None)) @ v.T
    d = np.sqrt(np.diag(c2))
    return c2 / np.outer(d, d)


def main():
    a = [x for x in sys.argv[1:] if not x.startswith("-")]
    s, e = (a[0], a[1]) if len(a) == 2 else (START_DATE, END_DATE)
    os.makedirs(OUT, exist_ok=True)
    data = load(s, e)
    if not data:
        raise SystemExit(f"[!] mse_*_{s}_{e}.csv файл олдсонгүй.")
    print(f"[+] {len(data)} хувьцаа ачааллаа: {', '.join(data)}")

    Q, L, R, D, W, rets = [], [], [], [], [], {}
    for t, df in data.items():
        bd = len(pd.bdate_range(df.index.min(), df.index.max()))
        Q.append(quality(t, df)); L.append(liquidity(t, df, bd))
        r = np.log(df["close"]).diff().dropna()
        r = r[r.abs() <= np.log(1 + JUMP)]          # хуваалт/алдааг хасна (01-ээс шалга)
        rets[t] = r
        if len(r) >= MIN_OBS:
            yrs = (df.index.max() - df.index.min()).days / 365.25
            R.append(returns_stats(t, r, yrs)); D.append(dependence(t, r)); W.append(weekday(t, r))

    pd.DataFrame(Q).to_csv(f"{OUT}/01_quality.csv", index=False)
    liq = pd.DataFrame(L).sort_values("median_value_mnt", ascending=False)
    liq.to_csv(f"{OUT}/02_liquidity.csv", index=False)
    pd.DataFrame(R).to_csv(f"{OUT}/03_returns.csv", index=False)
    pd.DataFrame(D).to_csv(f"{OUT}/04_dependence.csv", index=False)
    pd.DataFrame(W).to_csv(f"{OUT}/05_weekday.csv", index=False)

    # 6. синхрон корреляц: хоёулаа арилжаалагдсан өдрүүдээр (pairwise)
    big = [t for t in rets if len(rets[t]) >= MIN_OBS]
    panel = pd.DataFrame({t: rets[t] for t in big})
    corr = panel.corr(min_periods=100)
    corr.to_csv(f"{OUT}/06_corr.csv")
    C = corr.fillna(0).to_numpy(copy=True); np.fill_diagonal(C, 1)
    w = np.linalg.eigvalsh(C)
    print(f"[i] Корреляцын матрицын хамгийн бага хувийн утга: {w.min():.4f}"
          + ("  → эерэг хагас тодорхойлогдсон БИШ, засвар хийв" if w.min() < 0 else ""))
    Cp = nearest_psd(C) if w.min() < 0 else C

    # 7. PCA
    lam, V = np.linalg.eigh(Cp); lam, V = lam[::-1], V[:, ::-1]
    v1 = V[:, 0] * np.sign(V[:, 0].sum())
    pca = pd.DataFrame({"ticker": big, "pc1_loading": v1.round(3), "pc2_loading": V[:, 1].round(3)})
    pca.attrs["explained"] = lam / lam.sum()
    pca.to_csv(f"{OUT}/07_pca.csv", index=False)
    print(f"[i] PC1 нийт дисперсийн {100*lam[0]/lam.sum():.1f}%, PC2 {100*lam[1]/lam.sum():.1f}%")

    # 8. бета: тэнцүү жинтэй индекс, энгийн ба Димсоны (−1, 0, +1 хоцрогдол)
    # сийрэг арилжаа: хувьцааны өгөөж хэд хоногийг хамардаг тул индексийн өөрчлөлтийг
    # яг тэр хугацаанд нь нийлбэрлэж харьцуулна
    sel = (liq["median_value_mnt"] >= LIQ_VALUE) & (liq["pct_days_traded"] >= LIQ_DAYS)
    liquid = [t for t in liq.loc[sel, "ticker"] if t in big] or big[:5]
    print(f"[i] Индекс (тэнцүү жин, хөрвөх чадвартай): {', '.join(liquid)}")
    B = []
    for t in big:
        # тухайн хувьцааг индексээс хасна (өөрөөсөө регресслэх хазайлтаас сэргийлэх)
        members = [u for u in liquid if u != t]
        idx_lvl = panel[members].mean(axis=1, skipna=True).fillna(0).cumsum()
        y = panel[t].dropna()
        lv = idx_lvl.reindex(y.index)
        x0 = lv.diff()
        X = pd.concat([x0.shift(1), x0, x0.shift(-1)], axis=1)
        m = pd.concat([y, X], axis=1).dropna()
        if len(m) < MIN_OBS:
            continue
        b0 = np.polyfit(m.iloc[:, 2], m.iloc[:, 0], 1)[0]
        Xm = np.c_[np.ones(len(m)), m.iloc[:, 1:].values]
        coef = np.linalg.lstsq(Xm, m.iloc[:, 0].values, rcond=None)[0]
        B.append({"ticker": t, "beta_simple": round(b0, 3), "beta_dimson": round(coef[1:].sum(), 3),
                  "lag_share": round(coef[1] / coef[1:].sum(), 3) if coef[1:].sum() else np.nan,
                  "in_index": t in liquid})
    pd.DataFrame(B).to_csv(f"{OUT}/08_beta.csv", index=False)

    # график
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(liq["ticker"], liq["median_value_mnt"] / 1e6, color="#2f5d8a")
        ax.set_ylabel("Өдрийн медиан арилжаа, сая ₮"); ax.set_title("Хөрвөх чадвар")
        plt.xticks(rotation=60); plt.tight_layout(); plt.savefig(f"{OUT}/liquidity.png", dpi=150); plt.close()
        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(Cp, vmin=-1, vmax=1, cmap="RdBu_r")
        ax.set_xticks(range(len(big))); ax.set_xticklabels(big, rotation=90, fontsize=7)
        ax.set_yticks(range(len(big))); ax.set_yticklabels(big, fontsize=7)
        plt.colorbar(im); ax.set_title("Синхрон корреляц"); plt.tight_layout()
        plt.savefig(f"{OUT}/corr.png", dpi=150); plt.close()
    except Exception as ex:
        print(f"[!] График гаргасангүй: {ex}")

    pd.set_option("display.width", 200)
    print("\n=== ХӨРВӨХ ЧАДВАР (дээд 10) ===")
    print(liq.head(10).to_string(index=False))
    if D:
        print("\n=== ХАМААРАЛ ===")
        print(pd.DataFrame(D)[["ticker", "acf1", "ljungbox10_p", "acf1_abs", "VR5", "VR5_z"]].to_string(index=False))
    print(f"\n[+] Үр дүн: ./{OUT}/")


if __name__ == "__main__":
    main()

[+] 20 хувьцаа ачааллаа: AARD, APU, CUMN, GLMT, GOV, INV, KHAN, LEND, MGLA, MLG, MNDL, MSE, QPAY, SBM, SUU, TDB, TTL, TUM, UID, XAC
[i] Корреляцын матрицын хамгийн бага хувийн утга: 0.6960
[i] PC1 нийт дисперсийн 8.7%, PC2 6.4%
[i] Индекс (тэнцүү жин, хөрвөх чадвартай): KHAN, GLMT, MSE, APU, XAC, TUM

=== ХӨРВӨХ ЧАДВАР (дээд 10) ===
ticker  pct_days_traded  median_value_mnt  median_deals  amihud_x1e9  roll_spread_pct
  KHAN             94.8      95593476.000         137.0        0.034              NaN
  GLMT             94.9      22589622.105          64.0        0.168              NaN
   MSE             94.9      19706187.330          80.0        0.436              NaN
   APU             91.4      12213565.500          31.0        0.326             1.43
  AARD             58.4      12072796.000          36.0        0.606              NaN
   XAC             94.7       7680075.000          26.0        0.475             1.17
   TUM             94.7       5043339.735          49.0        

In [3]:
"""
МХБ: 2-р шатны шинжилгээ (mse_analysis.py-ийн дараа).
mse_history.py-ийн гаргасан mse_<T>_<start>_<end>.csv файлуудыг уншина.

Ажиллуулах:
    python mse_deep.py 2010-01-01 2026-10-14

Гаралт (./report2/):
    09_events.csv     |өгөөж|>30% өдрүүд + хувьцаа бүрийн хамгийн муу/сайн 5 өдөр
                      → ногдол ашиг, хуваалт, өгөгдлийн алдааг ГАРААР шалгах жагсаалт
    10_spreads.csv    Абди–Ранальдогийн спред (high/low/close-оос), Ролл, сүүлийн 12 сар
    11_epps.csv       1, 5, 20 хоногийн өгөөжөөрх дундаж корреляц ба PC1-ийн хувь
    12_reversal.csv   "Өчигдөр унасныг ав" (зөвхөн long) стратегийн бохир/цэвэр өгөөж
    13_garch.csv      GARCH(1,1): тогтвортой байдал α+β, хагас задралын хугацаа
    epps.png, reversal.png

Шаардлагатай сан: pandas numpy scipy matplotlib
"""
import glob
import os
import sys

import numpy as np
import pandas as pd
from scipy import optimize

START_DATE = "2010-01-01"
END_DATE = "2026-10-14"
JUMP = 0.30
MIN_OBS = 250
LIQ_VALUE = 5e6       # хөрвөх чадвартай: өдрийн медиан арилжаа ≥ 5 сая ₮
LIQ_DAYS = 0.90       # ба ажлын өдрийн ≥ 90%-д арилжаалагдсан
OUT = "report2"


def load(start, end):
    data = {}
    for fn in sorted(glob.glob(f"mse_*_{start}_{end}.csv")):
        t = os.path.basename(fn).split("_")[1]
        if t == "quality":
            continue
        df = pd.read_csv(fn, parse_dates=["date"]).sort_values("date")
        df = df.dropna(subset=["close"]).drop_duplicates("date")
        df = df[df["close"] > 0].set_index("date")
        if len(df) >= MIN_OBS:
            data[t] = df
    return data


# ---------------------------------------------------------------- 9. үйл явдал
def events(t, df):
    r = np.log(df["close"]).diff()
    gap = df.index.to_series().diff().dt.days
    tab = pd.DataFrame({"ret_pct": 100 * (np.exp(r) - 1), "gap_days": gap,
                        "close": df["close"], "prev_close": df["close"].shift(1),
                        "value_mnt": df["value"], "deals": df["deals"]}).dropna(subset=["ret_pct"])
    big = tab[np.abs(r.reindex(tab.index)) > np.log(1 + JUMP)].assign(kind="jump>30%")
    worst = tab.nsmallest(5, "ret_pct").assign(kind="worst5")
    best = tab.nlargest(5, "ret_pct").assign(kind="best5")
    out = pd.concat([big, worst, best])
    out = out[~out.index.duplicated()].reset_index().rename(columns={"index": "date"})
    out.insert(0, "ticker", t)
    num = out.select_dtypes("number").columns
    out[num] = out[num].round(2)
    return out


# ---------------------------------------------------------------- 10. спред
def abdi_ranaldo(df):
    """Abdi & Ranaldo (2017): s² = 4·E[(c_t − η_t)(c_t − η_{t+1})], η = (ln H + ln L)/2."""
    c = np.log(df["close"])
    eta = (np.log(df["high"]) + np.log(df["low"])) / 2
    x = (c - eta) * (c - eta.shift(-1))
    m = 4 * x.dropna().mean()
    return np.sqrt(m) if m > 0 else 0.0


def roll(df):
    dc = np.log(df["close"]).diff()
    cov = dc.cov(dc.shift(1))
    return 2 * np.sqrt(-cov) if cov < 0 else np.nan


def spreads(t, df):
    last = df[df.index >= df.index.max() - pd.Timedelta(days=365)]
    return {"ticker": t,
            "ar_spread_pct": round(100 * abdi_ranaldo(df), 2),
            "ar_spread_last12m_pct": round(100 * abdi_ranaldo(last), 2),
            "roll_spread_pct": round(100 * roll(df), 2),
            "roll_last12m_pct": round(100 * roll(last), 2)}


# ---------------------------------------------------------------- 11. Эппсийн нөлөө
def grid_prices(data):
    """Ажлын өдрийн тор дээрх лог үнэ: арилжаагүй өдөр өмнөх үнийг хадгална,
    бүртгэгдэхээс өмнө NaN. 30%-иас их үсрэлтийг (хуваалт/алдаа) тэглэнэ."""
    lo = min(d.index.min() for d in data.values())
    hi = max(d.index.max() for d in data.values())
    grid = pd.bdate_range(lo, hi)
    P = {}
    for t, df in data.items():
        r = np.log(df["close"]).diff().fillna(0)
        r[np.abs(r) > np.log(1 + JUMP)] = 0
        lp = r.cumsum()
        P[t] = lp.reindex(grid.union(lp.index)).ffill().reindex(grid)
        P[t][grid < df.index.min()] = np.nan
    return pd.DataFrame(P)


def avg_corr_pc1(R, min_p):
    C = R.corr(min_periods=min_p)
    keep = C.notna().sum() > 1
    C = C.loc[keep, keep]
    off = C.values[~np.eye(len(C), dtype=bool)]
    A = C.fillna(0).to_numpy(copy=True); np.fill_diagonal(A, 1)
    lam = np.clip(np.linalg.eigvalsh(A), 0, None)
    return np.nanmean(off), lam.max() / lam.sum(), len(C)


def epps(LP, groups):
    rows = []
    for name, cols in groups.items():
        for h in (1, 5, 20):
            R = LP[cols].iloc[::h].diff()              # давхцахгүй h хоногийн өгөөж
            R = R.where(R != 0)                         # хоёулаа хөдлөөгүй өдрийг хасна
            mc, pc1, k = avg_corr_pc1(R, max(30, 300 // h))
            rows.append({"universe": name, "horizon_days": h, "n_stocks": k,
                         "avg_pair_corr": round(mc, 3), "pc1_share": round(pc1, 3)})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- 12. эсрэг хөдөлгөөн
def reversal(LP, cols, spread):
    """t өдөр унасан (r_t < 0) хувьцааг t-ийн хаалтаар авч, t+1-ийн хаалтаар зарна.
    МХБ-д богино борлуулалт бараг боломжгүй тул зөвхөн long. Зардал = бүтэн спред
    (авахдаа хагас, зарахдаа хагас)."""
    R = LP[cols].diff()
    sig = (R < 0) & R.notna()
    nxt = R.shift(-1)
    w = sig.div(sig.sum(axis=1), axis=0)
    gross = (w * nxt).sum(axis=1, min_count=1)
    cost = (w * pd.Series(spread).reindex(cols)).sum(axis=1)
    ok = sig.sum(axis=1) > 0
    g, c = gross[ok].dropna(), cost[ok].reindex(gross[ok].dropna().index)
    n = (R[cols].notna().sum(axis=1) > 0).sum() / ((LP.index[-1] - LP.index[0]).days / 365.25)

    def summ(x, label):
        return {"series": label, "days": len(x),
                "mean_daily_bp": round(1e4 * x.mean(), 1),
                "t_stat": round(x.mean() / x.std() * np.sqrt(len(x)), 2),
                "ann_sharpe": round(x.mean() / x.std() * np.sqrt(n), 2)}
    return [summ(g, "gross"), summ(g - c, "net_of_spread"),
            {"series": "avg_cost_bp", "mean_daily_bp": round(1e4 * c.mean(), 1)}], g, g - c


# ---------------------------------------------------------------- 13. GARCH
def garch11(r):
    r = (r - r.mean()).values
    v0 = r.var()

    def nll(p):
        om, a, b = p
        if om <= 0 or a < 0 or b < 0 or a + b >= 0.999:
            return 1e10
        h = np.empty_like(r); h[0] = v0
        for i in range(1, len(r)):
            h[i] = om + a * r[i - 1] ** 2 + b * h[i - 1]
        return 0.5 * np.sum(np.log(h) + r ** 2 / h)
    best = None
    for a0, b0 in ((0.05, 0.9), (0.15, 0.7), (0.3, 0.3)):
        res = optimize.minimize(nll, [v0 * (1 - a0 - b0), a0, b0], method="Nelder-Mead",
                                options={"maxiter": 4000, "xatol": 1e-8, "fatol": 1e-8})
        if best is None or res.fun < best.fun:
            best = res
    return best.x


def main():
    a = [x for x in sys.argv[1:] if not x.startswith("-")]
    s, e = (a[0], a[1]) if len(a) == 2 else (START_DATE, END_DATE)
    os.makedirs(OUT, exist_ok=True)
    data = load(s, e)
    if not data:
        raise SystemExit(f"[!] mse_*_{s}_{e}.csv олдсонгүй.")

    ev = pd.concat([events(t, d) for t, d in data.items()], ignore_index=True)
    ev.to_csv(f"{OUT}/09_events.csv", index=False, date_format="%Y-%m-%d")

    sp = pd.DataFrame([spreads(t, d) for t, d in data.items()])
    sp.to_csv(f"{OUT}/10_spreads.csv", index=False)

    liquid = []
    for t, d in data.items():
        bd = len(pd.bdate_range(d.index.min(), d.index.max()))
        if d["value"].median() >= LIQ_VALUE and len(d) / bd >= LIQ_DAYS:
            liquid.append(t)
    print(f"[i] Хөрвөх чадвартай: {', '.join(liquid) or '—'}")

    LP = grid_prices(data)
    groups = {"all": list(data)}
    if len(liquid) >= 3:
        groups["liquid"] = liquid
    ep = epps(LP, groups)
    ep.to_csv(f"{OUT}/11_epps.csv", index=False)

    spread = (sp.set_index("ticker")["ar_spread_pct"] / 100).to_dict()
    rows, curves = [], {}
    for name, cols in groups.items():
        res, g, n = reversal(LP, cols, spread)
        for r_ in res:
            rows.append({"universe": name, **r_})
        curves[name] = (g, n)
    pd.DataFrame(rows).to_csv(f"{OUT}/12_reversal.csv", index=False)

    G = []
    for t in (liquid or list(data)[:5]):
        r = np.log(data[t]["close"]).diff().dropna()
        r = r[np.abs(r) <= np.log(1 + JUMP)]
        om, al, be = garch11(r)
        p = al + be
        G.append({"ticker": t, "omega": om, "alpha": round(al, 3), "beta": round(be, 3),
                  "persistence": round(p, 3), "at_bound": p >= 0.998,   # хязгаарт тулсан бол итгэх хэрэггүй
                  "half_life_days": round(np.log(0.5) / np.log(p), 1) if 0 < p < 1 else np.nan,
                  "uncond_ann_vol": round(np.sqrt(om / (1 - p) * 250), 3) if p < 1 else np.nan})
    pd.DataFrame(G).to_csv(f"{OUT}/13_garch.csv", index=False)

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(6, 4))
        for name, g in ep.groupby("universe"):
            ax.plot(g["horizon_days"], g["avg_pair_corr"], "o-", label=name)
        ax.set_xscale("log"); ax.set_xticks([1, 5, 20]); ax.set_xticklabels(["1 өдөр", "7 хоног", "сар"])
        ax.set_ylabel("Дундаж хос корреляц"); ax.set_title("Эппсийн нөлөө"); ax.legend()
        plt.tight_layout(); plt.savefig(f"{OUT}/epps.png", dpi=150); plt.close()
        fig, ax = plt.subplots(figsize=(7, 4))
        for name, (g, n) in curves.items():
            ax.plot(g.cumsum(), label=f"{name}: бохир")
            ax.plot(n.cumsum(), "--", label=f"{name}: спредийн дараа")
        ax.axhline(0, color="k", lw=0.5); ax.set_ylabel("Хуримтлагдсан лог өгөөж")
        ax.set_title("Өчигдөр унасныг ав (long-only)"); ax.legend(fontsize=8)
        plt.tight_layout(); plt.savefig(f"{OUT}/reversal.png", dpi=150); plt.close()
    except Exception as ex:
        print(f"[!] График гаргасангүй: {ex}")

    pd.set_option("display.width", 200)
    print("\n=== СПРЕД ===\n" + sp.to_string(index=False))
    print("\n=== ЭППС ===\n" + ep.to_string(index=False))
    print("\n=== ЭСРЭГ ХӨДӨЛГӨӨН ===\n" + pd.DataFrame(rows).to_string(index=False))
    print("\n=== GARCH ===\n" + pd.DataFrame(G).to_string(index=False))
    print(f"\n[+] ./{OUT}/ — 09_events.csv-г гараар шалга.")


if __name__ == "__main__":
    main()

[i] Хөрвөх чадвартай: APU, GLMT, KHAN, MSE, TUM, XAC

=== СПРЕД ===
ticker  ar_spread_pct  ar_spread_last12m_pct  roll_spread_pct  roll_last12m_pct
  AARD           0.00                   3.83              NaN              1.76
   APU           0.90                   0.77             1.43               NaN
  CUMN           2.68                   1.44             2.32               NaN
  GLMT           1.41                   2.06              NaN              0.05
   GOV           1.83                   1.91             1.76              0.42
   INV           2.40                   3.44             2.50              3.07
  KHAN           0.56                   0.00              NaN               NaN
  LEND           1.88                   1.51             1.14              0.70
  MGLA           2.60                   1.84             4.34              2.34
   MLG           2.94                   3.36             3.04              3.44
  MNDL           1.90                   2.10        

In [4]:
"""
МХБ: үнийн хязгаарт (±15%) хүрсэн өдрүүдийн үйл явдлын судалгаа (3-р шат).

Санаа: ханш тухайн өдрийн хязгаарт хүрвэл "жинхэнэ" үнэдээ нэг өдөрт хүрч чадахгүй,
тиймээс маргааш нь ижил чиглэлд үргэлжлэх (continuation) хандлагатай байж болно.
Хаалтын ханшаар авах боломжгүй (хязгаарт хүрсэн өдөр санал дуусдаг) тул ДАРААГИЙН
ӨДРИЙН НЭЭЛТЭЭР орж, k өдрийн дараах хаалтаар гарна гэж тооцно.

Ажиллуулах:
    python mse_limits.py 2010-01-01 2026-10-14
Гаралт (./report3/):
    14_limit_rule.csv     жил бүрийн |өгөөж|-ийн дээд утга, ±14.9%-аас дээш өдрийн тоо
    15_limit_runs.csv     дараалсан хязгаарын өдрүүдийн урт
    16_limit_events.csv   хязгаарын өдөр бүр: дараагийн нээлтийн цоорхой, k=1,2,5 өгөөж
    17_limit_summary.csv  дундаж, t, ялах магадлал; спредийн дараах цэвэр өгөөж
"""
import glob
import os
import sys

import numpy as np
import pandas as pd

START_DATE = "2010-01-01"
END_DATE = "2026-10-14"
LIMIT = 0.149          # |энгийн өгөөж| ≥ 14.9% бол хязгаарт хүрсэн гэж үзнэ
JUMP = 0.30
MIN_OBS = 250
LIQ_VALUE = 5e6
OUT = "report3"
H = (1, 2, 5)


def load(start, end):
    data = {}
    for fn in sorted(glob.glob(f"mse_*_{start}_{end}.csv")):
        t = os.path.basename(fn).split("_")[1]
        if t == "quality":
            continue
        df = pd.read_csv(fn, parse_dates=["date"]).sort_values("date")
        df = df.dropna(subset=["close"]).drop_duplicates("date")
        df = df[df["close"] > 0].set_index("date")
        if len(df) >= MIN_OBS:
            data[t] = df
    return data


def ar_spread(df):
    c = np.log(df["close"]); eta = (np.log(df["high"]) + np.log(df["low"])) / 2
    m = 4 * ((c - eta) * (c - eta.shift(-1))).dropna().mean()
    return np.sqrt(m) if m > 0 else 0.0


def main():
    a = [x for x in sys.argv[1:] if not x.startswith("-")]
    s, e = (a[0], a[1]) if len(a) == 2 else (START_DATE, END_DATE)
    os.makedirs(OUT, exist_ok=True)
    data = load(s, e)
    if not data:
        raise SystemExit(f"[!] mse_*_{s}_{e}.csv олдсонгүй.")

    rule, runs, ev = [], [], []
    for t, df in data.items():
        df = df.copy()
        r = df["close"].pct_change()
        ok = r.abs() <= JUMP                            # хуваалтын өдрийг хасна
        df["r"] = r.where(ok)
        df["year"] = df.index.year
        for y, g in df.groupby("year"):
            rr = g["r"].dropna()
            if len(rr):
                rule.append({"ticker": t, "year": y, "max_up_pct": round(100 * rr.max(), 2),
                             "max_dn_pct": round(100 * rr.min(), 2),
                             "n_ge_limit": int((rr.abs() >= LIMIT).sum()), "n_days": len(rr)})
        side = np.sign(df["r"]).where(df["r"].abs() >= LIMIT, 0).fillna(0)
        # дараалсан хязгаарын өдрүүд
        blk = (side != side.shift()).cumsum()
        for _, g in side.groupby(blk):
            if g.iloc[0] != 0:
                runs.append({"ticker": t, "start": g.index[0].date(), "side": int(g.iloc[0]),
                             "length": len(g)})
        sp = ar_spread(df)
        liquid = df["value"].median() >= LIQ_VALUE
        o, c = df["open"].values, df["close"].values
        idx = np.where(side.values != 0)[0]
        for i in idx:
            if i + 1 >= len(df):
                continue
            row = {"ticker": t, "date": df.index[i].date(), "side": int(side.iloc[i]),
                   "liquid": liquid, "spread": sp,
                   "close_eq_high": bool(df["close"].iloc[i] >= df["high"].iloc[i]),
                   "value_mnt": df["value"].iloc[i],
                   "gap_days_next": (df.index[i + 1] - df.index[i]).days,
                   # маргаашийн нээлт өнөөдрийн хаалтаас хэр зөрсөн бэ
                   "next_open_gap": o[i + 1] / c[i] - 1,
                   "cc_next": c[i + 1] / c[i] - 1}
            for k in H:
                j = i + k
                # дараагийн өдрийн нээлтээр орж, k-р арилжааны өдрийн хаалтаар гарна
                row[f"oc_{k}"] = c[j] / o[i + 1] - 1 if j < len(df) and o[i + 1] > 0 else np.nan
            ev.append(row)

    pd.DataFrame(rule).to_csv(f"{OUT}/14_limit_rule.csv", index=False)
    R = pd.DataFrame(runs)
    R.to_csv(f"{OUT}/15_limit_runs.csv", index=False)
    E = pd.DataFrame(ev)
    if E.empty:
        print("[!] Хязгаарт хүрсэн өдөр олдсонгүй."); return
    # ±30%-аас их = хуваалт, алдаа: хасна
    for k in H:
        E.loc[E[f"oc_{k}"].abs() > JUMP * 2, f"oc_{k}"] = np.nan
    E.to_csv(f"{OUT}/16_limit_events.csv", index=False)

    rows = []
    for (side, liq), g in E.groupby(["side", "liquid"]):
        for k in H:
            x = g[f"oc_{k}"].dropna() * side          # чиглэлээ дагасан өгөөж
            if len(x) < 10:
                continue
            net = x - g.loc[x.index, "spread"]          # авах+зарах = бүтэн спред
            rows.append({"side": "дээд хязгаар" if side > 0 else "доод хязгаар",
                         "liquid": liq, "hold_days": k, "n": len(x),
                         "mean_pct": round(100 * x.mean(), 2),
                         "t_stat": round(x.mean() / x.std() * np.sqrt(len(x)), 2),
                         "hit_rate": round((x > 0).mean(), 3),
                         "mean_net_pct": round(100 * net.mean(), 2),
                         "t_net": round(net.mean() / net.std() * np.sqrt(len(net)), 2),
                         "next_open_gap_pct": round(100 * (g["next_open_gap"] * side).mean(), 2)})
    S = pd.DataFrame(rows)
    S.to_csv(f"{OUT}/17_limit_summary.csv", index=False)

    pd.set_option("display.width", 200)
    yr = pd.DataFrame(rule).groupby("year")[["max_up_pct", "max_dn_pct"]].agg(["max", "min"])
    print("=== Жил бүрийн хамгийн их/бага өгөөж (хязгаар өөрчлөгдсөн эсэх) ===")
    print(yr.to_string())
    if len(R):
        print("\n=== Дараалсан хязгаарын урт ===")
        print(R.groupby("side")["length"].value_counts().sort_index().to_string())
    print("\n=== Хязгаарын дараах өгөөж (дараагийн нээлтээр орсон, чиглэлийг дагасан) ===")
    print(S.to_string(index=False))
    print("  side: дээд хязгаар → long; доод хязгаар → 'short' (МХБ-д бодитоор хийх боломжгүй, зөвхөн мэдээлэл)")
    print(f"\n[+] ./{OUT}/")


if __name__ == "__main__":
    main()

=== Жил бүрийн хамгийн их/бага өгөөж (хязгаар өөрчлөгдсөн эсэх) ===
     max_up_pct        max_dn_pct       
            max    min        max    min
year                                    
2012      14.90   5.52      -5.29 -11.06
2013      15.00   8.56      -6.71 -14.98
2014      15.00  11.97      -6.67 -15.00
2015      14.99  -3.38      -3.38 -15.00
2016      14.98   0.00       0.00 -14.96
2017      15.23  12.43       5.63 -14.96
2018      15.00   6.37      -4.92 -14.99
2019      14.71   6.68      -4.35 -12.81
2020      15.00   4.09      -4.24 -15.00
2021      15.00   4.29      -5.50 -14.99
2022      15.00   3.61      -6.70 -15.00
2023      15.00   4.12      -2.71 -14.98
2024      15.00   3.77      -4.02 -15.00
2025      15.00   3.43      -2.81 -15.00
2026      15.00   4.48      -2.56 -14.91

=== Дараалсан хязгаарын урт ===
side  length
-1    1         21
      2          2
 1    1         31
      2          7
      3          2
      4          1
      5          1
      9        